In [1]:
!pip -q install "pytorchvideo==0.1.5" "av" "decord"
!pip -q install ultralytics opencv-python


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
pip install numpy pandas tqdm scikit-learn opencv-python decord pytorchvideo imageio ffmpeg-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 349.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 196.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import cv2
import decord
from decord import VideoReader, cpu

from sklearn.metrics import balanced_accuracy_score, classification_report
from pytorchvideo.models.hub import x3d_s 

In [5]:
WORKSPACE = "/workspace"
DATA_ROOT = os.path.join(WORKSPACE, "datasets", "cctv_anomaly")
OUT_DIR   = os.path.join(WORKSPACE, "outputs", "x3d_threat14")
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

BEST_CKPT = os.path.join(OUT_DIR, "x3d_threat14_best_bacc.pth")

# Optional: keep Ultralytics cache/config under workspace too
os.environ["YOLO_CONFIG_DIR"] = os.path.join(WORKSPACE, ".config", "ultralytics")
os.makedirs(os.environ["YOLO_CONFIG_DIR"], exist_ok=True)

In [6]:
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| gpu:", torch.cuda.get_device_name(0) if device=="cuda" else "cpu")

decord.bridge.set_bridge("torch")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

device: cuda | gpu: NVIDIA A100-SXM4-80GB


In [7]:
pip install kagglehub


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import kagglehub
download_root = kagglehub.dataset_download("webadvisor/real-time-anomaly-detection-in-cctv-surveillance")


import shutil
if not os.path.exists(os.path.join(DATA_ROOT, "data")):
    print("Copying dataset to:", DATA_ROOT)
    shutil.copytree(download_root, DATA_ROOT, dirs_exist_ok=True)
else:
    print("Dataset already present at:", DATA_ROOT)

data_dir = os.path.join(DATA_ROOT, "data")
train_csv = os.path.join(data_dir, "train.csv")
test_csv  = os.path.join(data_dir, "test.csv")

train_df = pd.read_csv(train_csv)
test_df  = pd.read_csv(test_csv)

100%|██████████| 95.0G/95.0G [13:38<00:00, 125MB/s] 

Extracting files...


Copying dataset to: /workspace/datasets/cctv_anomaly


In [9]:
ALL_CLASSES = [
    "normal",
    "abuse", "arrest", "arson", "assault", "burglary", "explosion",
    "fighting", "roadaccidents", "robbery", "shooting", "shoplifting",
    "stealing", "vandalism",
]
class_to_id = {c: i for i, c in enumerate(ALL_CLASSES)}
id_to_class = {i: c for c, i in class_to_id.items()}

def norm_label(x): return str(x).strip().lower()

def build_abs_path(rel):
    rel = str(rel).replace("\n", "").replace("\\", "/").strip()
    return os.path.join(DATA_ROOT, rel)

def clean_df(df):
    items = []
    for _, r in df.iterrows():
        lbl = norm_label(r["label"])
        if lbl not in class_to_id:
            continue
        vp = build_abs_path(r["video_name"])
        if os.path.exists(vp):
            items.append((vp, class_to_id[lbl]))
    return items

train_items = clean_df(train_df)
test_items  = clean_df(test_df)
print("train items:", len(train_items), "test items:", len(test_items))

train items: 1520 test items: 380


In [11]:
CLIP_LEN = 48
SIZE = 224
BATCH = 16
NUM_WORKERS = 8  # try 12 if GPU underutilized and CPU has headroom

def kinetics_normalize(x_cthw: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor([0.45, 0.45, 0.45]).view(3,1,1,1)
    std  = torch.tensor([0.225,0.225,0.225]).view(3,1,1,1)
    return (x_cthw - mean) / std

def sample_clip_indices(total_frames, clip_len=48, mode="train"):
    if total_frames <= clip_len:
        return np.arange(total_frames)
    if mode == "train":
        start = random.randint(0, total_frames - clip_len)
    else:
        start = (total_frames - clip_len) // 2
    return np.arange(start, start + clip_len)

def load_clip_tchw(video_path, clip_len=48, size=224, mode="train"):
    vr = VideoReader(video_path, ctx=cpu(0))
    total = len(vr)

    idx = sample_clip_indices(total, clip_len=clip_len, mode=mode)
    batch = vr.get_batch(idx)
    frames = batch.cpu().numpy()

    if frames.shape[0] < clip_len:
        pad = clip_len - frames.shape[0]
        last = frames[-1:]
        frames = np.concatenate([frames, np.repeat(last, pad, axis=0)], axis=0)

    resized = [cv2.resize(fr, (size, size)) for fr in frames]

    if mode == "train":
        if random.random() < 0.5:
            resized = [cv2.flip(fr, 1) for fr in resized]
        if random.random() < 0.3:
            alpha = 0.9 + 0.2 * random.random()
            beta = random.randint(-10, 10)
            resized = [cv2.convertScaleAbs(fr, alpha=alpha, beta=beta) for fr in resized]

    frames = np.stack(resized, axis=0).astype(np.float32) / 255.0
    frames = np.transpose(frames, (3,0,1,2))

    x = torch.from_numpy(frames)
    x = kinetics_normalize(x)
    return x

class CCTVActionDataset(Dataset):
    def __init__(self, items, clip_len=48, size=224, mode="train"):
        self.items = items
        self.clip_len = clip_len
        self.size = size
        self.mode = mode

    def __len__(self): return len(self.items)

    def __getitem__(self, i):
        vp, y = self.items[i]
        x = load_clip_tchw(vp, clip_len=self.clip_len, size=self.size, mode=self.mode)
        return x, torch.tensor(y, dtype=torch.long)

train_ds = CCTVActionDataset(train_items, clip_len=CLIP_LEN, size=SIZE, mode="train")
test_ds  = CCTVActionDataset(test_items,  clip_len=CLIP_LEN, size=SIZE, mode="test")

train_labels = [y for _, y in train_items]
counts = Counter(train_labels)
class_weight_for_sampler = {c: 1.0 / np.sqrt(counts[c]) for c in counts}
sample_weights = torch.DoubleTensor([class_weight_for_sampler[y] for y in train_labels])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2
)

In [12]:
model = x3d_s(pretrained=True)

def replace_classifier_x3d(m, num_classes):
    if hasattr(m.blocks[-1], "proj") and isinstance(m.blocks[-1].proj, nn.Linear):
        in_f = m.blocks[-1].proj.in_features
        m.blocks[-1].proj = nn.Linear(in_f, num_classes)
        return
    linears = [(n, mod) for n, mod in m.named_modules() if isinstance(mod, nn.Linear)]
    last_name, last_lin = linears[-1]
    in_f = last_lin.in_features
    cur = m
    parts = last_name.split(".")
    for p in parts[:-1]:
        cur = getattr(cur, p)
    setattr(cur, parts[-1], nn.Linear(in_f, num_classes))

replace_classifier_x3d(model, len(ALL_CLASSES))
model = model.to(device)

criterion = nn.CrossEntropyLoss()

Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/X3D_S.pyth" to /root/.cache/torch/hub/checkpoints/X3D_S.pyth
100%|██████████| 29.4M/29.4M [00:00<00:00, 592MB/s]


In [13]:
def freeze_backbone_head_only():
    for p in model.parameters():
        p.requires_grad = False
    if hasattr(model.blocks[-1], "proj"):
        for p in model.blocks[-1].proj.parameters():
            p.requires_grad = True

def unfreeze_all():
    for p in model.parameters():
        p.requires_grad = True

freeze_backbone_head_only()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))
scheduler = None

/tmp/ipykernel_109/3898533564.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=="cuda"))


In [14]:
@torch.no_grad()
def evaluate_multiclip(clip_len=48, size=224):
    model.eval()
    total_loss, n = 0.0, 0
    all_true, all_pred = [], []

    for (vp, y) in tqdm(test_items, desc="eval-multiclip", leave=False):
        vr = VideoReader(vp, ctx=cpu(0))
        total = len(vr)

        if total <= clip_len:
            starts = [0]
        else:
            starts = [
                int(0.15 * (total - clip_len)),
                int(0.50 * (total - clip_len)),
                int(0.85 * (total - clip_len)),
            ]
            starts = [max(0, min(s, total - clip_len)) for s in starts]

        logits_sum = None
        for s in starts:
            idx = np.arange(s, min(s + clip_len, total))
            batch = vr.get_batch(idx)
            frames = batch.cpu().numpy()

            if frames.shape[0] < clip_len:
                pad = clip_len - frames.shape[0]
                last = frames[-1:]
                frames = np.concatenate([frames, np.repeat(last, pad, axis=0)], axis=0)

            resized = [cv2.resize(fr, (size, size)) for fr in frames]
            frames = np.stack(resized, axis=0).astype(np.float32) / 255.0
            frames = np.transpose(frames, (3,0,1,2))
            x = torch.from_numpy(frames)
            x = kinetics_normalize(x)

            x = x.unsqueeze(0).to(device)
            logits = model(x)
            logits_sum = logits if logits_sum is None else logits_sum + logits

        logits_avg = logits_sum / len(starts)
        yy = torch.tensor([y], device=device)
        loss = criterion(logits_avg, yy)

        total_loss += float(loss.item())
        n += 1
        all_true.append(y)
        all_pred.append(int(torch.argmax(logits_avg, dim=1).item()))

    acc = (np.array(all_pred) == np.array(all_true)).mean()
    bacc = balanced_accuracy_score(all_true, all_pred)
    return total_loss / n, acc, bacc, all_true, all_pred

In [15]:
EPOCHS = 25
best_bacc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(train_loader, desc=f"train epoch {epoch}/{EPOCHS}")

    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device=="cuda"), dtype=torch.float16):
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        pbar.set_postfix(loss=float(loss.item()))

    # After epoch 1: unfreeze & finetune LR, attach scheduler
    if epoch == 1:
        unfreeze_all()
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6, verbose=True
        )

    val_loss, val_acc, val_bacc, y_true, y_pred = evaluate_multiclip(clip_len=CLIP_LEN, size=SIZE)
    print(f"Epoch {epoch}: val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_bacc={val_bacc:.4f}")

    if scheduler is not None:
        scheduler.step(val_bacc)

    if val_bacc > best_bacc:
        best_bacc = val_bacc
        torch.save(model.state_dict(), BEST_CKPT)
        print("Saved best model (by bacc):", BEST_CKPT)

print("Best val bacc:", best_bacc)


train epoch 1/25:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_109/2339205398.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda"), dtype=torch.float16):
train epoch 1/25: 100%|██████████| 95/95 [00:49<00:00,  1.93it/s, loss=2.57]
/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1: val_loss=2.2549 val_acc=0.5211 val_bacc=0.1032
Saved best model (by bacc): /workspace/outputs/x3d_threat14/x3d_threat14_best_bacc.pth


train epoch 2/25:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_109/2339205398.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda"), dtype=torch.float16):
train epoch 2/25: 100%|██████████| 95/95 [01:12<00:00,  1.31it/s, loss=2.48]


Epoch 2: val_loss=2.2386 val_acc=0.5289 val_bacc=0.1299
Saved best model (by bacc): /workspace/outputs/x3d_threat14/x3d_threat14_best_bacc.pth


train epoch 3/25:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_109/2339205398.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda"), dtype=torch.float16):
train epoch 3/25: 100%|██████████| 95/95 [01:05<00:00,  1.45it/s, loss=2.25]


...

Epoch 23: val_loss=2.1312 val_acc=0.6342 val_bacc=0.2759


train epoch 24/25:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_109/2339205398.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda"), dtype=torch.float16):
train epoch 24/25: 100%|██████████| 95/95 [01:09<00:00,  1.36it/s, loss=1.94]


Epoch 24: val_loss=2.1230 val_acc=0.6474 val_bacc=0.2966
Saved best model (by bacc): /workspace/outputs/x3d_threat14/x3d_threat14_best_bacc.pth


train epoch 25/25:   0%|          | 0/95 [00:00<?, ?it/s]/tmp/ipykernel_109/2339205398.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=="cuda"), dtype=torch.float16):
train epoch 25/25: 100%|██████████| 95/95 [01:06<00:00,  1.44it/s, loss=2]   
                                                                 

Epoch 25: val_loss=2.1355 val_acc=0.6368 val_bacc=0.2783
Best val bacc: 0.2965538847117794


In [16]:
if os.path.exists(BEST_CKPT):
    model.load_state_dict(torch.load(BEST_CKPT, map_location=device))

val_loss, val_acc, val_bacc, y_true, y_pred = evaluate_multiclip(clip_len=CLIP_LEN, size=SIZE)
print("\nFINAL (best-by-bacc) val_loss:", val_loss)
print("FINAL val_acc:", val_acc, "FINAL val_bacc:", val_bacc)
print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=list(range(len(ALL_CLASSES))),
    target_names=ALL_CLASSES,
    zero_division=0
))

/tmp/ipykernel_109/3413484668.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(BEST_CKPT, map_location=device))
                         


FINAL (best-by-bacc) val_loss: 2.1230492115020754
FINAL val_acc: 0.6473684210526316 FINAL val_bacc: 0.2965538847117794

Classification report:
               precision    recall  f1-score   support

       normal       0.81      0.97      0.88       190
        abuse       0.38      0.60      0.46        10
       arrest       0.00      0.00      0.00        10
        arson       0.33      0.50      0.40        10
      assault       0.00      0.00      0.00        10
     burglary       0.31      0.25      0.28        20
    explosion       0.00      0.00      0.00        10
     fighting       0.15      0.20      0.17        10
roadaccidents       0.40      0.57      0.47        30
      robbery       0.55      0.57      0.56        30
     shooting       0.00      0.00      0.00        10
  shoplifting       0.00      0.00      0.00        10
     stealing       0.50      0.50      0.50        20
    vandalism       0.00      0.00      0.00        10

     accuracy                